In [1]:
import torch
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

%cd /kaggle/working
!git clone --depth 1 https://github.com/ultralytics/yolov5.git
%cd /kaggle/working/yolov5
!pip install -qr requirements.txt
!git rev-parse --short HEAD

Torch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4
/kaggle/working
Cloning into 'yolov5'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (164/164), done.
remote: Compressing objects: 100% (144/144), done.
remote: Total 164 (delta 50), reused 58 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (164/164), 984.97 KiB | 5.29 MiB/s, done.
Resolving deltas: 100% (50/50), done.
/kaggle/working/yolov5
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 9.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
b548e8d


In [2]:
from pathlib import Path
import yaml, zipfile, shutil

def find_dataset_roots(base):
    roots = []
    for yml in Path(base).rglob("data.yaml"):
        root = yml.parent
        ok = all((root / s / "images").exists() and (root / s / "labels").exists()
                 for s in ["train", "valid", "test"])
        if ok:
            roots.append(root)
    return roots

roots = find_dataset_roots("/kaggle/input/datasets/minhkhoathi/garbage-classification/GARBAGE CLASSIFICATION 3.v2-gc1.yolov5pytorch")

# Nếu bạn upload dạng .zip mà Kaggle chưa extract
if not roots:
    zips = list(Path("/kaggle/input").rglob("*.zip"))
    assert zips, "Không tìm thấy data.yaml hoặc file .zip trong /kaggle/input"
    extract_dir = Path("/kaggle/working/extracted_dataset")
    shutil.rmtree(extract_dir, ignore_errors=True)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zips[0], "r") as z:
        z.extractall(extract_dir)
    roots = find_dataset_roots(extract_dir)

preferred = [r for r in roots if "yolov5" in str(r).lower()]
DATA_ROOT = preferred[0] if preferred else roots[0]

with open(DATA_ROOT / "data.yaml", "r") as f:
    raw = yaml.safe_load(f)

names = raw["names"]
if isinstance(names, dict):
    names = [names[i] if i in names else names[str(i)] for i in range(len(names))]

NC = int(raw.get("nc", len(names)))

DATA = {
    "path": str(DATA_ROOT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": NC,
    "names": names,
}

DATA_YAML = Path("/kaggle/working/garbage_yolov5_data.yaml")
with open(DATA_YAML, "w") as f:
    yaml.safe_dump(DATA, f, sort_keys=False)

print("DATA_ROOT:", DATA_ROOT)
print(DATA_YAML.read_text())

DATA_ROOT: /kaggle/input/datasets/minhkhoathi/garbage-classification/GARBAGE CLASSIFICATION 3.v2-gc1.yolov5pytorch
path: /kaggle/input/datasets/minhkhoathi/garbage-classification/GARBAGE CLASSIFICATION
  3.v2-gc1.yolov5pytorch
train: train/images
val: valid/images
test: test/images
nc: 6
names:
- BIODEGRADABLE
- CARDBOARD
- GLASS
- METAL
- PAPER
- PLASTIC



In [3]:
from collections import Counter
import pandas as pd

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

summary, dist = [], []

for split in ["train", "valid", "test"]:
    img_dir = DATA_ROOT / split / "images"
    lab_dir = DATA_ROOT / split / "labels"

    images = [p for p in img_dir.iterdir() if p.suffix.lower() in IMG_EXTS]
    labels = list(lab_dir.glob("*.txt"))

    counter = Counter()
    for txt in labels:
        for line in txt.read_text().splitlines():
            if line.strip():
                counter[int(line.split()[0])] += 1

    summary.append({
        "split": split,
        "images": len(images),
        "label_files": len(labels),
        "boxes": sum(counter.values())
    })

    for cid, cname in enumerate(names):
        dist.append({
            "split": split,
            "class_id": cid,
            "class_name": cname,
            "boxes": counter[cid]
        })

display(pd.DataFrame(summary))
display(pd.DataFrame(dist))

,split,images,label_files,boxes
0,train,7324,7324,51611
1,valid,2098,2098,18916
2,test,1042,1042,3563


,split,class_id,class_name,boxes
0,train,0,BIODEGRADABLE,31721
1,train,1,CARDBOARD,3386
2,train,2,GLASS,5429
3,train,3,METAL,3948
4,train,4,PAPER,2981
5,train,5,PLASTIC,4146
6,valid,0,BIODEGRADABLE,13637
7,valid,1,CARDBOARD,1292
8,valid,2,GLASS,2380
9,valid,3,METAL,1360


In [4]:
%cd /kaggle/working/yolov5

IMG = 640
BATCH = 16      # nếu OOM thì giảm còn 8
EPOCHS = 50
MODEL = "yolov5s.pt"   # có thể đổi: yolov5n.pt, yolov5m.pt, yolov5l.pt, yolov5x.pt
RUN_NAME = "garbage_yolov5s"

!python train.py \
  --img {IMG} \
  --batch {BATCH} \
  --epochs {EPOCHS} \
  --data {DATA_YAML} \
  --weights {MODEL} \
  --project /kaggle/working/runs_yolov5 \
  --name {RUN_NAME} \
  --workers 2 \
  --patience 20 \
  --exist-ok

/kaggle/working/yolov5
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
WARNING ⚠️ wandb: wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: (30 second timeout) 
wandb: WARNING W&B disabled due to login timeout.
wandb: ERROR Error while calling W&B API: auth token too short (<Response [401]>)
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
train: weights=yolov5s.pt, cfg=, data=/kaggle/working/garbage_yolov5_data.yaml, hyp=data/hyps/hyp.scratch-low.yaml, epochs=50, batch_size=

In [5]:
import sys
from pathlib import Path
import pandas as pd

sys.path.append("/kaggle/working/yolov5")
import val

WEIGHTS = Path(f"/kaggle/working/runs_yolov5/{RUN_NAME}/weights/best.pt")
assert WEIGHTS.exists(), WEIGHTS

VAL_PROJECT = Path("/kaggle/working/runs_yolov5_val")
VAL_NAME = "garbage_test"

metrics, maps, times = val.run(
    data=str(DATA_YAML),
    weights=str(WEIGHTS),
    batch_size=BATCH,
    imgsz=IMG,
    task="test",
    conf_thres=0.001,
    iou_thres=0.6,
    max_det=300,
    save_txt=True,
    save_conf=True,
    project=VAL_PROJECT,
    name=VAL_NAME,
    exist_ok=True,
    plots=True,
    verbose=True,
)

mp, mr, map50, map5095 = metrics[:4]
official_df = pd.DataFrame([{
    "Precision": mp,
    "Recall": mr,
    "F1_from_P_R": 2 * mp * mr / (mp + mr + 1e-9),
    "mAP@0.5": map50,
    "mAP@0.5:0.95": map5095,
}])

per_class_official = pd.DataFrame({
    "class_id": range(NC),
    "class_name": names,
    "AP@0.5:0.95": maps[:NC],
})

display(official_df)
display(per_class_official)

YOLOv5 🚀 b548e8d Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)

Fusing layers... 
Model summary: 68 layers, 7,026,307 parameters, 0 gradients, 15.8 GFLOPs
test: Scanning /kaggle/input/datasets/minhkhoathi/garbage-classification/GARBAGE CLASSIFICATION 3.v2-gc1.yolov5pytorch/test/labels... 1042 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1042/1042 245.0it/s 4.3s
WARNING ⚠️ test: Cache directory /kaggle/input/datasets/minhkhoathi/garbage-classification/GARBAGE CLASSIFICATION 3.v2-gc1.yolov5pytorch/test is not writeable: [Errno 30] Read-only file system: '/kaggle/input/datasets/minhkhoathi/garbage-classification/GARBAGE CLASSIFICATION 3.v2-gc1.yolov5pytorch/test/labels.cache.npy'
                 Class     Images  Instances          P          R      mAP50   mAP50-95: 100% ━━━━━━━━━━━━ 66/66 2.8it/s 23.6s
                   all       1042       3563      0.516      0.407      0.413      0.274
         BIODEGRADABLE       1042         49      0.182      0.286      0.

,Precision,Recall,F1_from_P_R,mAP@0.5,mAP@0.5:0.95
0,0.516361,0.406942,0.455168,0.413292,0.274159


,class_id,class_name,AP@0.5:0.95
0,0,BIODEGRADABLE,0.078728
1,1,CARDBOARD,0.052950
2,2,GLASS,0.274159
3,3,METAL,0.474886
4,4,PAPER,0.374836
5,5,PLASTIC,0.389394


In [6]:
import numpy as np
import pandas as pd
from pathlib import Path

IOU_MATCH = 0.50
CONF_FOR_PRF1 = 0.25
IOU_THRESHOLDS = np.round(np.arange(0.50, 0.96, 0.05), 2)

GT_DIR = DATA_ROOT / "test" / "labels"
PRED_DIR = VAL_PROJECT / VAL_NAME / "labels"
assert PRED_DIR.exists(), PRED_DIR

def read_yolo_txt(path, pred=False):
    cols = 6 if pred else 5
    if not path.exists() or path.stat().st_size == 0:
        return np.zeros((0, cols), dtype=float)
    arr = np.loadtxt(path, ndmin=2).astype(float)
    if pred and arr.shape[1] == 5:
        arr = np.c_[arr, np.ones(len(arr))]
    return arr

def xywh_to_xyxy(x):
    y = x.copy()
    y[:, 0] = x[:, 0] - x[:, 2] / 2
    y[:, 1] = x[:, 1] - x[:, 3] / 2
    y[:, 2] = x[:, 0] + x[:, 2] / 2
    y[:, 3] = x[:, 1] + x[:, 3] / 2
    return np.clip(y, 0, 1)

def box_iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)
    inter = wh[:, :, 0] * wh[:, :, 1]
    area_a = np.clip(a[:, 2] - a[:, 0], 0, None) * np.clip(a[:, 3] - a[:, 1], 0, None)
    area_b = np.clip(b[:, 2] - b[:, 0], 0, None) * np.clip(b[:, 3] - b[:, 1], 0, None)
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)

image_stems = sorted(
    {p.stem for p in (DATA_ROOT / "test" / "images").iterdir() if p.suffix.lower() in IMG_EXTS}
    | {p.stem for p in GT_DIR.glob("*.txt")}
    | {p.stem for p in PRED_DIR.glob("*.txt")}
)

GT, PRED = {}, {}

for stem in image_stems:
    g = read_yolo_txt(GT_DIR / f"{stem}.txt", pred=False)
    p = read_yolo_txt(PRED_DIR / f"{stem}.txt", pred=True)

    GT[stem] = (
        g[:, 0].astype(int),
        xywh_to_xyxy(g[:, 1:5])
    ) if len(g) else (np.array([], dtype=int), np.zeros((0, 4)))

    PRED[stem] = (
        p[:, 0].astype(int),
        xywh_to_xyxy(p[:, 1:5]),
        p[:, 5]
    ) if len(p) else (np.array([], dtype=int), np.zeros((0, 4)), np.array([]))

def compute_ap(recall, precision):
    if len(recall) == 0:
        return 0.0
    mrec = np.r_[0.0, recall, 1.0]
    mpre = np.r_[1.0, precision, 0.0]
    mpre = np.maximum.accumulate(mpre[::-1])[::-1]
    x = np.linspace(0, 1, 101)
    return float(np.trapz(np.interp(x, mrec, mpre), x))

def collect_class(cid):
    gts, preds, n_gt = {}, [], 0

    for stem in image_stems:
        gc, gb = GT[stem]
        pc, pb, ps = PRED[stem]

        gboxes = gb[gc == cid]
        gts[stem] = gboxes
        n_gt += len(gboxes)

        for box, score in zip(pb[pc == cid], ps[pc == cid]):
            preds.append((stem, float(score), box))

    preds.sort(key=lambda x: x[1], reverse=True)
    return gts, preds, n_gt

def match_predictions(gts, preds, iou_thr, conf_min=None):
    if conf_min is not None:
        preds = [p for p in preds if p[1] >= conf_min]

    used = {stem: np.zeros(len(boxes), dtype=bool) for stem, boxes in gts.items()}
    tp = np.zeros(len(preds))
    fp = np.zeros(len(preds))
    matched_ious = []

    for i, (stem, score, box) in enumerate(preds):
        boxes = gts.get(stem, np.zeros((0, 4)))

        if len(boxes):
            ious = box_iou_matrix(box[None, :], boxes)[0]
            best_idx = int(np.argmax(ious))
            best_iou = float(ious[best_idx])
        else:
            best_idx, best_iou = -1, 0.0

        if best_iou >= iou_thr and best_idx >= 0 and not used[stem][best_idx]:
            tp[i] = 1
            used[stem][best_idx] = True
            matched_ious.append(best_iou)
        else:
            fp[i] = 1

    return tp, fp, matched_ious

def evaluate_at_iou(iou_thr):
    rows = []
    total_tp = total_fp = total_fn = 0
    all_ious, ap_values = [], []

    for cid, cname in enumerate(names):
        gts, preds, n_gt = collect_class(cid)

        tp_all, fp_all, _ = match_predictions(gts, preds, iou_thr)
        rec_curve = np.cumsum(tp_all) / (n_gt + 1e-9)
        prec_curve = np.cumsum(tp_all) / (np.cumsum(tp_all) + np.cumsum(fp_all) + 1e-9)
        ap = compute_ap(rec_curve, prec_curve) if n_gt > 0 else np.nan

        tp_fixed, fp_fixed, matched_ious = match_predictions(
            gts, preds, iou_thr, conf_min=CONF_FOR_PRF1
        )

        tp = int(tp_fixed.sum())
        fp = int(fp_fixed.sum())
        fn = int(n_gt - tp)

        precision = tp / (tp + fp + 1e-9)
        recall = tp / (tp + fn + 1e-9)
        f1 = 2 * precision * recall / (precision + recall + 1e-9)
        mean_iou = float(np.mean(matched_ious)) if matched_ious else 0.0

        rows.append({
            "class_id": cid,
            "class_name": cname,
            "gt_boxes": n_gt,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "mean_matched_IoU": mean_iou,
            "Precision": precision,
            "Recall": recall,
            "F1-score": f1,
            "AP": ap,
        })

        if n_gt > 0:
            ap_values.append(ap)

        total_tp += tp
        total_fp += fp
        total_fn += fn
        all_ious.extend(matched_ious)

    precision = total_tp / (total_tp + total_fp + 1e-9)
    recall = total_tp / (total_tp + total_fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)

    summary = {
        "IoU_threshold": iou_thr,
        "confidence_for_P/R/F1": CONF_FOR_PRF1,
        "mean_matched_IoU": float(np.mean(all_ious)) if all_ious else 0.0,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "mAP": float(np.nanmean(ap_values)),
    }

    return pd.DataFrame(rows), summary

per_class_50, summary_50 = evaluate_at_iou(IOU_MATCH)

map_curve = []
for t in IOU_THRESHOLDS:
    _, s = evaluate_at_iou(float(t))
    map_curve.append({"IoU_threshold": float(t), "mAP": s["mAP"]})

map_curve = pd.DataFrame(map_curve)
summary_50["mAP@0.5:0.95_custom"] = map_curve["mAP"].mean()

display(per_class_50)
display(pd.DataFrame([summary_50]))
display(map_curve)

per_class_50.to_csv("/kaggle/working/yolov5_test_per_class_metrics_iou50.csv", index=False)
pd.DataFrame([summary_50]).to_csv("/kaggle/working/yolov5_test_summary_metrics.csv", index=False)
map_curve.to_csv("/kaggle/working/yolov5_test_map_by_iou.csv", index=False)

/tmp/ipykernel_23/3343509509.py:71: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(np.interp(x, mrec, mpre), x))


,class_id,class_name,gt_boxes,tp,fp,...,mean_matched_IoU,Precision,Recall,F1-score,AP
0,0,BIODEGRADABLE,49,19,121,...,0.706614,0.135714,0.387755,0.201058,0.155368
1,1,CARDBOARD,20,6,126,...,0.935989,0.045455,0.300000,0.078947,0.060640
2,2,GLASS,0,0,169,...,0.000000,0.000000,0.000000,0.000000,NaN
3,3,METAL,533,330,133,...,0.875738,0.712743,0.619137,0.662651,0.642016
4,4,PAPER,1376,627,260,...,0.847528,0.706877,0.455669,0.554132,0.563033
5,5,PLASTIC,1585,965,655,...,0.811045,0.595679,0.608833,0.602184,0.645430


,IoU_threshold,confidence_for_P/R/F1,mean_matched_IoU,Precision,Recall,F1-score,mAP,mAP@0.5:0.95_custom
0,0.5,0.25,0.833125,0.5708,0.54645,0.55836,0.413298,0.274165


,IoU_threshold,mAP
0,0.50,0.413298
1,0.55,0.396215
2,0.60,0.375002
3,0.65,0.353206
4,0.70,0.323691
5,0.75,0.283115
6,0.80,0.245064
7,0.85,0.202545
8,0.90,0.125636
9,0.95,0.023877
